# ComfyUI Inference

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

FOLDERNAME = 'training_zen/phase2/assignment2_4'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content/drive/My\ Drive/$FOLDERNAME

In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt
!pip install -q IPython==8.18.1
%load_ext autoreload
%autoreload 2

In [ ]:
# 3. Clone ComfyUI repository if not already cloned
import os
if not os.path.exists("ComfyUI"):
    print("Cloning ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    print("ComfyUI directory already exists.")

# Install ComfyUI requirements
!pip install -r ComfyUI/requirements.txt

In [ ]:
# 4. Create the ComfyUI Custom Node to load our custom StableDiffusion model & LoRA weights
import os

custom_node_dir = "ComfyUI/custom_nodes"
os.makedirs(custom_node_dir, exist_ok=True)
custom_node_path = os.path.join(custom_node_dir, "custom_sd_node.py")

node_code = """import os
import sys
import torch
import yaml
import numpy as np

# Dynamically find the workspace directory relative to this custom node
# custom_node is in ComfyUI/custom_nodes/custom_sd_node.py
# So the parent's parent is the workspace directory
workspace_path = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
if workspace_path not in sys.path:
    sys.path.insert(0, workspace_path)
    sys.path.insert(0, os.path.join(workspace_path, "dreamboth-lora-trainer"))

from models.stable_diffusion import StableDiffusion
from src.model_utils import inject_lora
from safetensors.torch import load_file

class CustomStableDiffusionSampler:
    @classmethod
    def INPUT_TYPES(s):
        return {
            "required": {
                "prompt": ("STRING", {"multiline": True, "default": "a photo of sks person"}),
                "negative_prompt": ("STRING", {"multiline": True, "default": ""}),
                "steps": ("INT", {"default": 50, "min": 1, "max": 10000}),
                "cfg_scale": ("FLOAT", {"default": 7.5, "min": 0.0, "max": 100.0, "step": 0.1}),
                "seed": ("INT", {"default": 42, "min": 0, "max": 0xffffffffffffffff}),
                "width": ("INT", {"default": 512, "min": 64, "max": 2048, "step": 64}),
                "height": ("INT", {"default": 512, "min": 64, "max": 2048, "step": 64}),
            }
        }

    RETURN_TYPES = ("IMAGE",)
    FUNCTION = "generate"
    CATEGORY = "Custom Stable Diffusion"

    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32
        
        print("[Custom SD Node] Loading custom StableDiffusion model...")
        self.sd = StableDiffusion()
        self.sd.load_weight(
            model_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
            device=self.device,
            quantization="fp16" if self.device == "cuda" else "fp32"
        )
        
        # Inject LoRA config
        print("[Custom SD Node] Injecting LoRA architecture...")
        config_path = os.path.join(workspace_path, "dreamboth-lora-trainer", "training_config.yaml")
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        self.sd.unet, self.sd.text_encoder = inject_lora(self.sd.unet, self.sd.text_encoder, config)
        
        # Load LoRA weights
        lora_weights_path = os.path.join(workspace_path, "checkpoints", "final_lora_weights.safetensors")
        if not os.path.exists(lora_weights_path):
            lora_weights_path = os.path.join(workspace_path, "checkpoints", "final_lora_weights.bin")
            
        if os.path.exists(lora_weights_path):
            print(f"[Custom SD Node] Loading LoRA weights from {lora_weights_path}...")
            if lora_weights_path.endswith(".safetensors"):
                state_dict = load_file(lora_weights_path)
            else:
                state_dict = torch.load(lora_weights_path, map_location="cpu")
                
            unet_state_dict = {k.replace("unet.", ""): v for k, v in state_dict.items() if k.startswith("unet.")}
            text_state_dict = {k.replace("text_encoder.", ""): v for k, v in state_dict.items() if k.startswith("text_encoder.")}
            
            self.sd.unet.load_state_dict(unet_state_dict, strict=False)
            if len(text_state_dict) > 0 and self.sd.text_encoder is not None:
                self.sd.text_encoder.load_state_dict(text_state_dict, strict=False)
            print("[Custom SD Node] LoRA weights loaded successfully!")
        else:
            print("[Custom SD Node] WARNING: LoRA weights not found. Running base model inference.")
            
    def generate(self, prompt, negative_prompt, steps, cfg_scale, seed, width, height):
        # Set seed
        generator = torch.Generator(device=self.device)
        generator.manual_seed(seed)
        
        # Sample noise
        latent_width = width // 8
        latent_height = height // 8
        noise = torch.randn(
            (1, 4, latent_height, latent_width),
            generator=generator,
            device=self.device,
            dtype=self.dtype
        )
        
        # Run forward pass
        print(f"[Custom SD Node] Generating image for prompt: '{prompt}'...")
        with torch.inference_mode():
            images_tensor = self.sd(
                prompts=[prompt],
                noise=noise,
                num_steps=steps,
                device=self.device,
                guidance_scale=cfg_scale,
                negative_prompts=[negative_prompt]
            )
            
            # Post-process to ComfyUI format (scale range [-1, 1] to [0, 1] and permute to B, H, W, C)
            images = (images_tensor / 2 + 0.5).clamp(0, 1)
            images = images.permute(0, 2, 3, 1).to(torch.float32).cpu()
            
        return (images,)

NODE_CLASS_MAPPINGS = {
    "CustomStableDiffusionSampler": CustomStableDiffusionSampler
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "CustomStableDiffusionSampler": "Custom Stable Diffusion Sampler"
}
"""

with open(custom_node_path, "w", encoding="utf-8") as f:
    f.write(node_code)

print(f"Custom node successfully written to {custom_node_path}")

## 5. Set up ngrok Authtoken

To expose ComfyUI publicly, you need a free ngrok account. Get your token from the [ngrok Dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) and enter it below.

In [ ]:
# Enter your ngrok authtoken
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"  # @param {type:"string"}

from pyngrok import ngrok
if NGROK_TOKEN and NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN":
    ngrok.set_auth_token(NGROK_TOKEN)
    print("ngrok token set successfully!")
else:
    print("WARNING: Please enter a valid ngrok token to start the server.")

## 6. Start ngrok tunnel & Launch ComfyUI

Execute the cell below to start the ngrok tunnel on port 8188 and run ComfyUI. Click on the printed **Public URL** link to open the ComfyUI web UI in a new tab.

In [ ]:
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Open ngrok tunnel
try:
    public_url = ngrok.connect(8188)
    print("*" * 60)
    print(f"ComfyUI Public URL: {public_url.public_url}")
    print("*" * 60)
except Exception as e:
    print(f"Error connecting to ngrok: {e}")

# Run ComfyUI server
%cd ComfyUI
!python main.py --listen --port 8188